## ## Package Imports & Common Variables

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import plotly.express as px
import datetime

load_dotenv()
ticker = ''
path_stockdata = os.path.join(os.path.join(os.environ.get('judgement_day'), 'Data--StockWatchList'), ticker)
path_analysis_csv = os.path.join(path_stockdata, f'{ticker}--Analysis_Margins-s1v1.csv')

today = datetime.date.today()
cutoff = today.year - 20
cutoff10 = today.year - 10
cutoff5 = today.year - 5
cutoff3 = today.year - 3

## Data Imports & Cleaning

In [ ]:
%%capture

df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--Annual_10K-s1v1.csv'), index_col=0)
df0 = df0.dropna(subset=['FiscalYear'])
df0[['FiscalYear', 'FiscalMonth']] = df0[['FiscalYear', 'FiscalMonth']].astype(int)


In [ ]:
df0

## Profit Margin


In [ ]:
%%capture

profit_df = df0[['FiscalYear', 'Revenue', 'GrossProfit', 'OperatingIncome', 'NetIncome']]
profit_df['GPM'] = round(profit_df['GrossProfit'] / profit_df['Revenue'] * 100, 2)
profit_df['OPM'] = round(profit_df['OperatingIncome'] / profit_df['Revenue'] * 100, 2)
profit_df['NPM'] = round(profit_df['NetIncome'] / profit_df['Revenue'] * 100, 2)

In [ ]:
profit_df.tail(10)

In [ ]:
profit_fig = px.bar(profit_df.tail(10), y=['GPM', 'OPM', 'NPM'], x='FiscalYear', barmode='group', orientation='v',
                    title='10 Year Profit Margins', template='plotly_dark',
                    color_discrete_map={'GPM': 'blue',
                                        'OPM': 'yellow',
                                        'NPM': 'green'
                                        })
profit_fig.update_layout(yaxis_title='Margin %')
profit_fig.show()

In [ ]:
gpm_mean = round((profit_df['GPM'].tail(10).mean()) , 0)
gpm_median = round((profit_df['GPM'].tail(10).median()) , 0)

opm_mean = round((profit_df['OPM'].tail(10).mean()), 0)
opm_median = round((profit_df['OPM'].tail(10).median()) , 0)

npm_mean = round((profit_df['NPM'].tail(10).mean()), 0)
npm_median = round((profit_df['NPM'].tail(10).median()), 0)

print(f'10 Year Mean Gross Profit Margin: {gpm_mean}%')
print(f'10 Year Median Gross Profit Margin: {gpm_median}%')

print(f'10 Year Mean Operating Profit Margin: {opm_mean}%')
print(f'10 Year Median Operating Profit Margin: {opm_median}%')

print(f'10 Year Mean Net Profit Margin: {npm_mean}%')
print(f'10 Year Median Net Profit Margin: {npm_median}%')

## Cash Margins

In [ ]:
%%capture

cash_df = df0[['FiscalYear', 'Revenue', 'OpCash', 'FreeCash', 'CAPEX']]
cash_df['OCM'] = round(cash_df['OpCash'] / cash_df['Revenue'] * 100, 2)
cash_df['FCM'] = round(cash_df['FreeCash'] / cash_df['Revenue'] * 100 , 2)
cash_df['CAPEXM'] = round(cash_df['CAPEX'] / cash_df['Revenue'] * 100, 2).abs()

In [ ]:
cash_fig = px.bar(cash_df.tail(10), y=['OCM', 'FCM', 'CAPEXM'], x='FiscalYear', barmode='group', orientation='v',
                    title='10 Cash Margins', template='plotly_dark',
                    color_discrete_map={'OCM': 'blue',
                                        'FCM': 'green',
                                        'CAPEXM': 'red'
                                        })
cash_fig.update_layout(yaxis_title='Margin %')
cash_fig.show()

In [ ]:
ocm_mean = round((cash_df['OCM'].tail(10).mean()) , 0)
ocm_median = round((cash_df['OCM'].tail(10).median()), 0)

fcm_mean = round((cash_df['FCM'].tail(10).mean()), 0)
fcm_median = round((cash_df['FCM'].tail(10).median()), 0)

capexm_mean = round((cash_df['CAPEXM'].tail(10).mean()) , 0)
capexm_median = round((cash_df['CAPEXM'].tail(10).median()) , 0)

print(f'10 Year Mean Operating Cash Margin: {ocm_mean}%')
print(f'10 Year Median Operating Cash Margin: {ocm_median}%')

print(f'10 Year Mean Free Cash Margin: {fcm_mean}%')
print(f'10 Year Median OFree Cash  Margin: {fcm_median}%')

print(f'10 Year Mean CAPEX Margin: {capexm_mean}%')
print(f'10 Year Median CAPEX Margin: {capexm_median}%')

## Analysis Output

In [ ]:
metrics_json = {
    "type": "margins",
    "date": today.strftime('%Y-%m-%d'),
    "profit_margin_data_type": "numeric percentage",
    "gpm_mean_10y": gpm_mean,
    "gpm_median_10y": gpm_median,
    "opm_mean_10y": opm_mean,
    "opm_median_10y": opm_median,
    "npm_mean_10y": npm_mean,
    "npm_median_10y": npm_median,
    "ocm_mean_10y": ocm_mean,
    "ocm_median_10y": ocm_median,
    "fcm_mean_10y": fcm_mean,
    "fcm_median_10y": fcm_median,
    "capexm_mean_10y": capexm_mean,
    "capexm_median_10y": capexm_median
}

metrics_json

In [ ]:
metrics_df = pd.DataFrame([metrics_json])
metrics_df

In [ ]:
if os.path.isfile(path_analysis_csv):
    metrics_df.to_csv(path_analysis_csv, mode='a', header=False, index=False)
else:
    metrics_df.to_csv(path_analysis_csv, mode='w', header=True, index=False)

## End Notebook